# CMSE 381 Final Project Template

**INSTRUCTIONS**: This is a template to help organize your project.  All projects should include the 5 major sections below (you do not need to use this template file).  If you use this file, complete your work below and remove content in parentheses. Also, remove this current cell.  

#### CMSE 381 Final Project
### &#9989; Group members: Abigail Parisot, Hunter Windham
### &#9989; Section_002
#### &#9989; Due Friday, December 5th

# An Investigation into the Freiwald Tsao Face AM Dataset

## Background and Motivation

In analyzing this dataset, we sought to conclude how neural activation data predicts the identity of individuals. We set out to create a model that can be trained on the Freiwald Tsao dataset to accurately predict what face image a monkey was looking at, given the activation of various neurons.

## Methodology
_(How did you go about answering your question(s)? You should wrote some code here to demonstrate what the data is like and how in principle your method works. You can leave the variations of the related to specific results to the results section.)_

To answer our questions, we looked at the MSE and test accuracy of various models. We performed linear and ridge regression, as well as logistic and ridge regression. Linear regression was tested as a baseline for predicting the activation patterns, with ridge to handle multicolinearity. Logistic regression was chosen to classify the person labels because it handles multi-class problems. We also built a neural network to attempt to intepret the neuron activation as the monkey's brain would.

In [1]:
# you may want to import some modules here
import pandas as pd

### Data
_(Describe the data you are using. What variables are you using? What they mean? Why did you choose them?)_

This data describes two different monkeys neurological reactions to seeing different images of faces. The monkeys recognize the different faces and they are categorized as (person) 1 - 25. Initally we import the data and concatonate it vertically. 

In [4]:
# Code inspired by
#  https://www.geeksforgeeks.org/python/importing-multiple-files-in-python/
import os

folder_path = "Data2"
files = os.listdir(folder_path)
datas = [] # master list of dfs

for file in files: # save each file to master list
    if file.endswith(".csv"):
        file_path = os.path.join(folder_path, file)
        df = pd.read_csv(file_path).iloc[:,:406] # only use the first 400 miliseconds
        
        df['neuron'] = file[-7:-4]
        move = df.pop('neuron')  
        df.insert(0, 'neuron', move) 

        datas.append(df)
        

df = pd.concat(datas, axis=0, ignore_index=True)
print(df.shape)
df.head()

(206216, 407)


,neuron,site_info.monkey,site_info.region,labels.stimID,labels.person,labels.orientation,labels.orient_person_combo,time.1_2,time.2_3,time.3_4,...,time.391_392,time.392_393,time.393_394,time.394_395,time.395_396,time.396_397,time.397_398,time.398_399,time.399_400,time.400_401
0,013,bert,am,1,1,front,front 1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,013,bert,am,1,1,front,front 1,0,0,0,...,0,0,0,0,0,0,0,0,0,1
2,013,bert,am,1,1,front,front 1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,013,bert,am,2,2,front,front 2,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,013,bert,am,2,2,front,front 2,0,1,0,...,0,0,0,0,0,0,0,0,0,0


### What are the different columns?
#### site_info.monkey: 
This column is what monkey the observation came from.
#### site_info.region:
The face selective region (only anterior medial exists in this dataset)
#### labels.person
This column labels which person the image is taken of.
#### labels.orientation
This is the direction the persons face is oriented in the image. 
#### time.1_2 ... time.800_801
Each csv file denotes a specific neuron. For each neuron, there are 800 miliseconds of data. In the data, a 1 represents an active neuron and a 0, inactive. 

After some consideration, we decided neuron activation a feature, so we modified the data frame to correspond to this. 

In [5]:
weighted = pd.DataFrame({
    "activation_weight": df.iloc[:,7:406].sum(axis=1)
})
averaged_data = pd.concat([df.iloc[:,0:7],weighted], axis=1)
grouped = averaged_data.groupby(["neuron", 'labels.person', 'labels.orientation'], 
                                as_index = False)['activation_weight'].sum()
cleaned = grouped.pivot_table(
    index=['labels.person', 'labels.orientation'],
    columns='neuron',
    values='activation_weight'
)

cleaned.head()

neuron                             005  009   013  014  017  018   021   025  \
labels.person labels.orientation                                               
1             back                29.0  4.0  12.0  1.0  1.0  2.0  23.0  20.0   
              down                31.0  5.0  38.0  3.0  6.0  4.0   8.0  16.0   
              front               48.0  0.0  26.0  2.0  2.0  1.0   5.0  13.0   
              left 3/4            38.0  3.0  42.0  0.0  0.0  1.0  15.0  13.0   
              left profile        23.0  2.0  16.0  0.0  4.0  5.0  12.0  22.0   

neuron                            026   029  ...   401   402  405   409  413  \
labels.person labels.orientation             ...                               
1             back                1.0  24.0  ...   4.0   1.0  3.0  21.0  7.0   
              down                1.0  29.0  ...   6.0   6.0  1.0  11.0  9.0   
              front               0.0  32.0  ...  14.0  15.0  4.0  12.0  5.0   
              left 3/4            0.0  22.0  ...   6.0   5.0  3.0  32.0  5.0   
              left profile        1.0  31.0  ...   1.0   5.0  5.0  20.0  9.0   

neuron                            421   425   429  433   437  
labels.person labels.orientation                              
1             back                8.0   4.0   5.0  5.0  36.0  
              down                4.0  21.0  10.0  1.0  26.0  
              front               7.0   8.0   4.0  3.0  20.0  
              left 3/4            6.0  11.0   6.0  1.0  40.0  
              left profile        6.0   5.0   4.0  4.0  41.0  

[5 rows x 155 columns]

### Other methods used _(if applicable)_

_(If this is a preprocessing step to prepare your data for regression or classification models, you should put this subsection before your explanation for the regression or classification models.)_

_(What method did you use otherwise? Why did you choose to use them? What questions would you answer with them? How would you evaluate the results? What cross-validation method did you use when applicable?)_

All models used principle component analysis to select the features which contribute the most to the output. To process this, we selected the top 50 neurons with the most activation, and from those found 10 principle components. 

In [6]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

In [7]:
sums = cleaned.sum(axis = 0)
top_neurons = sums.sort_values(ascending=False).head(50).index
top_features = cleaned[top_neurons]
X = StandardScaler().fit_transform(top_features)
pca = PCA(n_components=10)  
X_pca = pca.fit_transform(X)

### Models for classification _(if applicable)_
_(What models will you be using for classification? Why did you choose to use them? What questions would you answer with them? How would you evaluate if each model? What cross-validation method did you use?)_

In [ ]:
# you may add some code here to show how the model works in principle

### Models for regression _(if applicable)_
_(What models will you be using for regression? Why did you choose to use them? What questions would you answer with them? How would you evaluate if each model? What cross-validation method did you use?)_

In [ ]:
# you may add some code here to show how the model works in principle

# you may add some code here to show how the model works in principle

## Results

_(What did you find when you carried out your methods? Some of your code related to
presenting results/figures/data may be replicated from the methods section or may only be present in
this section. All of the plots that you plan on using for your presentation should be present in this
section)_

### classification results
_(What are you trying to do here?)_

In [ ]:
# how did you do it

_(How do you interpret what you see?)_

_(What are you doing next?)_

In [ ]:
# how did you do it (etc. etc.)

### regression results
_(What are you trying to do here?)_

In [ ]:
# how did you do it

_(How do you interpret what you see?)_

_(What are you doing next?)_

In [ ]:
# how did you do it (etc. etc.)

### other results
_(What are you trying to do here?)_

In [ ]:
# how did you do it

_(How do you interpret what you see?)_

_(What are you doing next?)_

In [ ]:
# how did you do it (etc. etc.)

## Discussion and Conclusion

_(What did you learn from your results? What obstacles did you run into? What would you do differently next time? Clearly provide quantitative answers to your question(s)?  At least one of your questions should be answered with numbers.  That is, it is not sufficient to answer "yes" or "no", but rather to say something quantitative such as variable 1 increased roughly 10% for every 1 year increase in variable 2.)_

### discussion on the classification results

### discussion on the regression results

### discussion on the other results

### conclusion and future steps

## Author contribution

_(Please describe the contribution of each member of group)._

## References

_(List the source(s) for any data and/or literature cited in your project.  Ideally, this should be formatted using a formal citation format (MLA or APA or other, your choice!).   Multiple free online citation generators are available such as <a href="http://www.easybib.com/style">http://www.easybib.com/style</a>. **Important:** if you use **any** code that you find on the internet for your project you **must** cite it or you risk losing most/all of the points for you project.)_

- https://www.geeksforgeeks.org/python/importing-multiple-files-in-python
- Freiwald, W. A., & Tsao, D. Y. (2010). Functional compartmentalization and viewpoint generalization within the macaque face-processing system. Science, 330(6005), 845-851.
- Meyers, E. M., Borzello, M., Freiwald, W. A., & Tsao, D. (2015). Intelligent information loss: The coding of facial identity, head pose, and non-face information in the macaque face patch system. Journal of Neuroscience, 35(18).